In [1]:
import pandas as pd
import numpy as np
import unicodedata

def describir_dataset(df, titulo):
    """
    Imprime métricas clave para entender el estado de un DataFrame.

    """
    print(f"\n--- {titulo} del Dataset ---")
    print("Primeras 5 filas:")
    print(df.head())
    print("\nInformación del DataFrame:")
    df.info()
    print("\nEstadísticas descriptivas:")
    print(df.describe())
    
    # Conteo de valores faltantes
    missing_values = df.isnull().sum()
    missing_percentage = (missing_values / len(df)) * 100
    missing_df = pd.DataFrame({'Missing Count': missing_values, 'Missing %': missing_percentage})
    print("\nValores Faltantes por Columna:")
    print(missing_df[missing_df['Missing Count'] > 0].sort_values(by='Missing %', ascending=False))
    
    # Conteo de duplicados
    num_duplicates = df.duplicated().sum()
    print(f"\nNúmero de filas duplicadas: {num_duplicates}")

def limpiar_nombres_columnas(df):
    """
    Limpia y estandariza los nombres de las columnas de un DataFrame.
    
    """
    print("Iniciando la limpieza de nombres de columnas...")
    new_columns = []
    for col in df.columns:
        cleaned_name = col.strip().lower()
        cleaned_name = unicodedata.normalize('NFKD', cleaned_name).encode('ascii', 'ignore').decode('utf-8')
        cleaned_name = cleaned_name.replace('(', '').replace(')', '')
        cleaned_name = cleaned_name.replace('°', '')
        cleaned_name = cleaned_name.replace('ug/m3', 'ug_m3')
        cleaned_name = cleaned_name.replace('%', 'percent')
        cleaned_name = cleaned_name.replace('w/m2', 'w_m2')
        cleaned_name = cleaned_name.replace('m/s', 'm_s')
        cleaned_name = cleaned_name.replace('mmhg', 'mm_hg')
        cleaned_name = cleaned_name.replace(' ', '_')
        cleaned_name = cleaned_name.replace('.', '_')
        cleaned_name = cleaned_name.replace('/', '_')
        cleaned_name = cleaned_name.replace('__', '_').replace('__', '_').strip('_')
        new_columns.append(cleaned_name)
        
    df.columns = new_columns
    print("Nombres de columnas estandarizados.")
    return df

def manejar_datos_faltantes(df):
    """
    Maneja los valores faltantes en un DataFrame, eliminando primero las
    columnas con 100% de valores nulos y luego imputando los restantes.

    """
    print("Iniciando el manejo de datos faltantes...")
    df_processed = df.copy()
    
    # --- Paso 1: Eliminar columnas con 100% de valores faltantes ---
    total_rows = len(df_processed)
    cols_to_drop = [col for col in df_processed.columns if df_processed[col].isnull().sum() == total_rows]
    if cols_to_drop:
        df_processed.drop(columns=cols_to_drop, inplace=True)
        print(f"Se eliminaron las siguientes columnas con 100% de valores faltantes: {', '.join(cols_to_drop)}")
    else:
        print("No se encontraron columnas con 100% de valores faltantes.")

    # --- Paso 2: Manejar duplicados ---
    initial_rows = df_processed.shape[0]
    df_processed.drop_duplicates(inplace=True)
    rows_after_duplicates = df_processed.shape[0]
    if initial_rows - rows_after_duplicates > 0:
        print(f"Se eliminaron {initial_rows - rows_after_duplicates} filas duplicadas.")

    # --- Paso 3: Imputar valores faltantes restantes ---
    for col in df_processed.columns:
        if df_processed[col].isnull().any():
            if pd.api.types.is_numeric_dtype(df_processed[col]):
                median_val = df_processed[col].median()
                df_processed[col] = df_processed[col].fillna(median_val)
                print(f"Valores faltantes en '{col}' (numérica) imputados con la mediana: {median_val:.2f}")
            elif df_processed[col].dtype == 'object':
                df_processed[col] = df_processed[col].fillna('DESCONOCIDO')
                print(f"Valores faltantes en '{col}' (categórica) imputados con 'DESCONOCIDO'.")
    
    return df_processed

def transformar_datos(df):
    """
    Realiza transformaciones de datos 
    """
    print("Iniciando la transformación")
    df_transformed = df.copy()

    # Conversión de la columna de fecha y hora
    if 'fecha_y_hora' in df_transformed.columns:
        df_transformed['fecha_y_hora'] = pd.to_datetime(df_transformed['fecha_y_hora'], errors='coerce', infer_datetime_format=True)
        df_transformed.dropna(subset=['fecha_y_hora'], inplace=True)
        print("Columna 'fecha_y_hora' convertida a tipo datetime.")

        # Creación de nuevas características a partir de la fecha
        df_transformed['año'] = df_transformed['fecha_y_hora'].dt.year
        df_transformed['mes'] = df_transformed['fecha_y_hora'].dt.month
        df_transformed['dia'] = df_transformed['fecha_y_hora'].dt.day
        df_transformed['hora'] = df_transformed['fecha_y_hora'].dt.hour
        print("Columnas de tiempo (año, mes, día, hora) creadas.")
    
    return df_transformed

def main():
    """
    Función principal que orquesta el proceso de preparación de datos.
    """
    file_path = 'Datasets/SISTEMA_DE_VIGILANCIA_DE_CALIDAD_DE_AIRE.csv'
    
    try:
        df = pd.read_csv(file_path, sep=',', na_values=['NA', '-999'])
    except FileNotFoundError:
        print(f"Error: El archivo '{file_path}' no se encontró.")
        return
        
    # --- ETAPA 1: Estado Inicial (Sucio) ---
    describir_dataset(df, "Estado Inicial")
    
    # --- ETAPA 2: Limpieza de Columnas y Manejo de Datos Faltantes ---
    df_limpio = limpiar_nombres_columnas(df.copy())
    df_limpio = manejar_datos_faltantes(df_limpio)
    
    # --- ETAPA 3: Estado Intermedio ---
    describir_dataset(df_limpio, "Estado Intermedio (Datos Limpios)")
    
    # --- ETAPA 4: Transformación y Feature Engineering ---
    df_final = transformar_datos(df_limpio)
    
    # --- ETAPA 5: Estado Final (Listo para el Análisis) ---
    describir_dataset(df_final, "Estado Final (Listo para el Análisis)")
    
    print("\nProceso de preparación de datos finalizado con éxito.")
    print(f"Dataset final con {df_final.shape[0]} filas y {df_final.shape[1]} columnas.")

if __name__ == "__main__":
    main()


--- Estado Inicial del Dataset ---
Primeras 5 filas:
             FECHA Y HORA  Presión atmosférica (mmHg)  \
0  01/01/2024 12:00:00 AM                     696.291   
1  01/01/2024 01:00:00 AM                     696.132   
2  01/01/2024 02:00:00 AM                     695.799   
3  01/01/2024 03:00:00 AM                     695.300   
4  01/01/2024 04:00:00 AM                     694.951   

   Temperatura Ambiente (Celsius)  Velocidad del Viento (m/s)  \
0                          24.086                       1.142   
1                          23.734                       1.420   
2                          23.169                       1.592   
3                          23.281                       1.368   
4                          22.887                       2.341   

   Dirección del viento (°)  Humedad Relativa (%)  Radiación Solar (W/m²)  \
0                   181.289                83.830                     0.0   
1                   121.161                82.638         

C:\Users\Diego\AppData\Local\Temp\ipykernel_2432\3460162452.py:103: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  df_transformed['fecha_y_hora'] = pd.to_datetime(df_transformed['fecha_y_hora'], errors='coerce', infer_datetime_format=True)
C:\Users\Diego\AppData\Local\Temp\ipykernel_2432\3460162452.py:103: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_transformed['fecha_y_hora'] = pd.to_datetime(df_transformed['fecha_y_hora'], errors='coerce', infer_datetime_format=True)


Columna 'fecha_y_hora' convertida a tipo datetime.
Columnas de tiempo (año, mes, día, hora) creadas.

--- Estado Final (Listo para el Análisis) del Dataset ---
Primeras 5 filas:
         fecha_y_hora  presion_atmosferica_mm_hg  \
0 2024-01-01 00:00:00                    696.291   
1 2024-01-01 01:00:00                    696.132   
2 2024-01-01 02:00:00                    695.799   
3 2024-01-01 03:00:00                    695.300   
4 2024-01-01 04:00:00                    694.951   

   temperatura_ambiente_celsius  velocidad_del_viento_m_s  \
0                        24.086                     1.142   
1                        23.734                     1.420   
2                        23.169                     1.592   
3                        23.281                     1.368   
4                        22.887                     2.341   

   direccion_del_viento  humedad_relativa_percent  radiacion_solar_w_m2  \
0               181.289                    83.830                  